# From a code on paper to a runnable qodec

A quantum error correcting code, as it appears in a paper, is a short list of
Pauli operators: the stabilizers that define the codespace, and the operators
that represent the logical qubits. That is enough to reason about the code, and
nowhere near enough to *run* it. Running it needs circuits — how to prepare an
encoded state, how to hold it, how to read it back — and every one of those
circuits has to be written, checked, and kept in sync with the code.

`qdk.ec.develop.qodec_from_code` does that step for you. Hand it a
`qodec.Code` and it returns a complete, verified, runnable
[qodec](https://github.com/microsoft/qodec): a logical instruction set over the
code's logical qubits, lowering to physical stim operations, with a synthesized
circuit behind every instruction.

This notebook takes the Steane code from its stabilizers to a sampled memory
experiment without writing a single circuit by hand.

## Installing

```bash
pip install "qdk[ec,ec-backends]"
```


## 1. The code, as you would write it down

The Steane [[7,1,3]] code: seven physical qubits, one logical qubit, distance 3.
Six stabilizer generators — three X-type, three Z-type — and one logical X / Z
pair. This is the whole input.

In [ ]:
import qodec

steane = qodec.Code(
    "steane",
    stabilizers=[
        "X_0 X_3 X_4 X_6",
        "X_1 X_3 X_5 X_6",
        "X_2 X_4 X_5 X_6",
        "Z_0 Z_3 Z_4 Z_6",
        "Z_1 Z_3 Z_5 Z_6",
        "Z_2 Z_4 Z_5 Z_6",
    ],
    x=["X_0 X_1 X_3"],
    z=["Z_1 Z_2 Z_5"],
)

print(f"{len(list(steane.stabilizers))} stabilizers, {len(list(steane.x))} logical qubit(s)")

## 2. Synthesis

One call turns that into a runnable qodec.

In [ ]:
from qdk.ec import audit, develop, profile, targets
from qdk.ec.develop import qodec_from_code, synthesis_notes

codec = qodec_from_code(steane)
print(codec.summary())

The result is a two-layer qodec. The top layer is a *synthesized* logical ISA —
instructions that talk about the logical qubit, not the seven physical ones —
and the bottom layer is the physical stim ISA the gadgets lower into.

In [ ]:
logical = codec.layers[0]

for mnemonic, instruction in sorted(logical.isa.instructions.items()):
    print(f"{mnemonic:12s} {instruction.description}")

## 3. The circuits it wrote

`idle` is a syndrome-extraction round: one ancilla per stabilizer, each prepared
in |+>, coupled to its stabilizer's support with a controlled Pauli, then
rotated back and measured.

Note that `CX` is used where the stabilizer has an X, and `CZ` where it has a Z.
That one uniform construction handles CSS and non-CSS codes alike, and no data
qubit is ever touched by a basis-changing gate.

In [ ]:
print(logical.gadgets["idle"].circuit.source)

Readout is transversal, and the logical Pauli gadgets are just the code's own
logical operators applied gate by gate.

In [ ]:
for mnemonic in ("prepare_z", "measure_z", "measure_x", "x0", "z0"):
    source = logical.gadgets[mnemonic].circuit.source.strip().replace("\n", " ; ")
    print(f"{mnemonic:12s} {source[:78]}")

## 4. What makes it trustworthy

Synthesis does not assert that its circuits are right — it *proves* it, twice
over, and keeps only what passes.

First, checks and readouts are never hand-derived. Each circuit is emitted as a
draft and `complete_gadget` discovers, by exact simulation, which parities of
measurement outcomes are deterministic (the checks a decoder consumes) and which
carry the logical answer (the readouts).

In [ ]:
idle = logical.gadgets["idle"]

print(f"{len(idle.checks)} checks discovered for `idle`; the first two:")
for check in list(idle.checks)[:2]:
    print("   ", [str(atom) for atom in check])

Second, every finished gadget is checked against the instruction it claims to
implement: the action its circuit *realizes* must equal the action the
instruction *declares*. Anything that fails is dropped rather than shipped, so a
gadget that survives is one whose circuit provably does what it says.

In [ ]:
mismatches = {
    mnemonic: profile.gadget_action_mismatch(gadget)
    for mnemonic, gadget in logical.gadgets.items()
    if profile.gadget_action_mismatch(gadget) is not None
}
print("gadgets whose circuit disagrees with its declared action:", mismatches or "none")

The code's distance survives the trip, and the full audit runs over the
synthesized qodec exactly as it would over a hand-authored one.

In [ ]:
distance, witness = profile.code_distance_of(codec.codes["steane"])
print("code distance:", distance, "| witness:", [str(p) for p in witness])

report = audit.audit(codec)
print(f"audit: {len(report.errors())} error(s), {len(report.warnings())} warning(s)")
for diagnostic in report.errors():
    print("   ", diagnostic.rule, "|", diagnostic.summary)

> **A note on that error.** The `gadget/readout-mismatch` rule misfires on
> X-basis destructive measurement gadgets: it also fires on the hand-authored
> `c4` qodec that ships with `qdk.ec`, and it fires asymmetrically on `measure_x`
> but not `measure_z` for codes like Steane that are perfectly X/Z symmetric. It
> is a property of that audit rule, not of the synthesized circuit — the
> declared-vs-realized action check above passes for every gadget.

## 5. Running it

The qodec is immediately usable by every `qdk.ec` target. Here is a memory
experiment written entirely in logical instructions.

In [ ]:
from qodec.circuits import Program


def call(mnemonic: str) -> qodec.instructions.InstructionCall:
    instruction = logical.isa.instruction(mnemonic)
    inputs = {str(i): "q" for i in range(len(list(instruction.inputs)))}
    outputs = {str(i): "q" for i in range(len(list(instruction.outputs)))}
    if not inputs and not outputs:
        return qodec.instructions.InstructionCall(mnemonic)
    return qodec.instructions.InstructionCall(mnemonic, inputs=inputs, outputs=outputs)


program = Program(
    [call(m) for m in ("prepare_z", "idle", "idle", "measure_z")], logical.isa
)
print([c.mnemonic for c in program.instructions])

Noiseless, no detector may fire. If one does, the qodec is wrong.

In [ ]:
import numpy as np

noiseless = targets.StimSampler(codec)
shots = np.asarray(noiseless.execute(program, shots=256))
events = noiseless.emitter.detection_events(program, shots)

print(f"{shots.shape[0]} shots x {shots.shape[1]} measurement records")
print(f"{events.shape[1]} detectors, {int(events.sum())} fired")

With noise, they fire — the synthesized syndrome extraction is doing real work.

In [ ]:
noisy = targets.StimSampler(codec, noise={"p_data": 0.01, "p_meas": 0.01})
noisy_shots = np.asarray(noisy.execute(program, shots=2000))
fired = noisy.emitter.detection_events(program, noisy_shots).any(axis=1)

print(f"shots with at least one detection: {fired.mean():.1%}")

And the detector error model a decoder would consume falls out of the same
qodec.

In [ ]:
dem = targets.detector_error_model_of(
    codec, program, {"p_data": 0.001, "p_meas": 0.001}
)
print("\n".join(str(dem).splitlines()[:6]))

model = targets.depolarizing(0.001)
gadget_distance, _ = targets.gadget_distance_of(logical.gadgets["idle"], model)
print("\ncircuit-level distance of `idle`:", gadget_distance)

### The circuits are textbook, not fault-tolerant

That last number is worth dwelling on. The *code* has distance 3, but the
synthesized `idle` gadget has circuit-level distance 1: a single fault can cause
an undetected logical error.

This is not a defect in the synthesis — it is a true property of the construction
it uses. Extracting a stabilizer with one unflagged ancilla means a single fault
on that ancilla, midway through its string of controlled Paulis, propagates onto
several data qubits at once. Fault-tolerant extraction needs more: flag qubits,
Shor- or Steane-style ancilla preparation, or a code-specific schedule — all of
which are design decisions a general synthesizer should not silently make for
you.

So read `qodec_from_code` as what it is: the fastest path from a code to
something you can *run and measure*, and a correct baseline to compare a
hand-tuned, fault-tolerant qodec against. `targets.gadget_distance_of` is
exactly the instrument for telling the two apart.

## 6. Deploying it

The synthesized qodec is ordinary data — it serializes, round-trips, and is the
artifact you hand to a compilation pipeline. Nothing about it is second-class
compared to a hand-written one.

In [ ]:
text = develop.to_yaml(codec)
restored = develop.from_yaml(text)

print(f"{len(text.splitlines())} lines of YAML")
print("round-trips:", sorted(restored.layers[0].gadgets) == sorted(logical.gadgets))

## 7. When synthesis cannot finish the job

Not every instruction exists for every code, and `qodec_from_code` will not
pretend otherwise. Take the five-qubit code as it is conventionally written,
with a logical Z that carries X components.

In [ ]:
FIVE_QUBIT_STABILIZERS = [
    "Z_0 X_1 X_2 Z_3",
    "Z_1 X_2 X_3 Z_4",
    "Z_0 Z_2 X_3 X_4",
    "X_0 Z_1 Z_3 X_4",
]

as_written = qodec.Code(
    "five_qubit",
    stabilizers=list(FIVE_QUBIT_STABILIZERS),
    x=["X_0 X_1 X_2 X_3 X_4"],
    z=["X_0 X_3 Z_4"],
)

partial = qodec_from_code(as_written)
print("synthesized:", sorted(partial.layers[0].gadgets))
for mnemonic, reason in synthesis_notes(partial)["omitted"].items():
    print(f"  omitted {mnemonic:12s} {reason[:88]}")

`prepare_z` resets the data qubits to |0...0> and projects into the codespace,
which pins the logical state only when the code's logical Z is a Z-type
operator. Here it is not, so no such gadget exists — and rather than emit a
circuit that quietly prepares the wrong state, synthesis omits it and says why.

The qodec it does return is still coherent: it only advertises instructions it
can actually lower.

In [ ]:
print("instructions:", sorted(partial.layers[0].isa.instructions))
print("gadgets:     ", sorted(partial.layers[0].gadgets))

Strikingly, the omission is a property of *how the code was written down*, not
of the code itself. The same five-qubit code with an all-Z logical Z — an
equally valid choice from the same coset — synthesizes more of the menu.

In [ ]:
all_z = qodec.Code(
    "five_qubit_all_z",
    stabilizers=list(FIVE_QUBIT_STABILIZERS),
    x=["X_0 X_1 X_2 X_3 X_4"],
    z=["Z_0 Z_1 Z_2 Z_3 Z_4"],
)

better = qodec_from_code(all_z)
print("as written :", sorted(partial.layers[0].gadgets))
print("all-Z basis:", sorted(better.layers[0].gadgets))

Pass `strict=True` when a partial qodec is not acceptable and you would rather
be told immediately.

In [ ]:
try:
    qodec_from_code(as_written, strict=True)
except ValueError as error:
    print("strict=True raised:", str(error)[:120])

## Where to go next

* `qodec_from_code(code, name=..., description=..., strict=...)` — synthesis.
* `synthesis_notes(codec)` — what was built, and what was omitted and why.
* `qdk.ec.develop` — `complete_gadget` / `complete_qodec` finish hand-written
  drafts the same way synthesis finishes generated ones.
* `qdk.ec.profile` and `qdk.ec.audit` — characterize and verify the result.
* `qdk.ec.targets` — sample it, build detector error models, estimate distance.

See `qdk_ec_walkthrough.ipynb` for the full develop / test / deploy lifecycle on
a hand-authored qodec.